# Tutorial 01: OpenContinualEnv Quickstart (Hugging Face OpenEnv Native)

Welcome to **OpenContinualEnv**! This interactive notebook walks through creating a standardized Hugging Face `openenv` continuous learning environment, executing actions in the safe Python sandbox, capturing interaction trajectories, evaluating rewards, and testing learning controllers.

## 1. Environment Instantiation

First, we import `OpenContinualEnv` and `OpenContinualAction` from `open_continual_env`.

In [ ]:
import os
from open_continual_env.env.core_env import OpenContinualEnv, OpenContinualAction

# Create OpenEnv environment instance
env = OpenContinualEnv()
obs, info = env.reset()

print(f"Task ID: {obs.task_id}")
print(f"Prompt : {obs.prompt}")

## 2. Environment Step & Python Sandbox Execution

Now we pass a Python code action to the environment. The environment executes the code safely inside `PythonSandbox` and computes the multi-objective reward.

In [ ]:
# Define code action
action_code = """
def add(a, b):
    return a + b
"""

# Step environment using OpenEnv action
action = OpenContinualAction(code=action_code)
obs, reward, terminated, truncated, info = env.step(action)

print(f"Execution Success : {info.get('success')}")
print(f"Unit Test Pass Rate: {info.get('pass_rate')}")
print(f"Computed Reward   : {reward:.3f}")

## 3. Learning Controller Routing Decision

The `LearningController` evaluates step performance and routes the trajectory to `IGNORE`, `STORE_MEMORY`, `UPDATE_LORA`, or `UPDATE_BASE`.

In [ ]:
from open_continual_env.controller import LearningController
from open_continual_env.trajectory.schema import Trajectory

controller = LearningController()
traj = Trajectory(
    trajectory_id="demo_traj_01",
    prompt=obs.prompt,
    model_response=action_code,
    reasoning_notes="",
    generated_code=action_code,
    execution_output=info,
    feedback={},
    reward=reward,
    regression_results={},
    timestamp=""
)

decision, d_info = controller.decide(traj)
print(f"Controller Decision: {decision}")
print(f"Decision Metadata : {d_info}")

## 4. Experience Store Serialization

Finally, we store the interaction trajectory into the thread-safe `ExperienceStore`.

In [ ]:
from open_continual_env.trajectory.store import ExperienceStore

store = ExperienceStore()
store.add(traj)
print(f"Total Trajectories Saved: {len(store)}")